In [1]:
!conda env export > environment.yml
!pip install transformers datasets scikit-learn
!pip install torch torchvision torchaudio
!pip install tensorflow
!pip install hf_xet
!pip install tqdm
!pip install evaluate
!pip install "accelerate>=0.26.0"
!pip install ipywidgets
!pip install huggingface_hub

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
import transformers
import numpy as np
from transformers import pipeline
import torch
import tensorflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    BartForSequenceClassification,
    BartTokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
import evaluate 
import transformers 

In [2]:
# Function to extract text from HTML
def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

# Load the csv file
# df = pd.read_csv('D:/comp-6713-industry-project/raw_data/seniority_labelled_development_set.csv')
# df_test = pd.read_csv('D:/comp-6713-industry-project/raw_data/seniority_labelled_test_set.csv')
df = pd.read_csv('raw_data/seniority_labelled_development_set.csv')
df_test = pd.read_csv('raw_data/seniority_labelled_test_set.csv')


In [3]:
apostrophe_dict = {
"ain't": "am not / are not",
"aren't": "are not / am not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he had / he would",
"he'd've": "he would have",
"he'll": "he shall / he will",
"he'll've": "he shall have / he will have",
"he's": "he has / he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how has / how is",
"i'd": "I had / I would",
"i'd've": "I would have",
"i'll": "I shall / I will",
"i'll've": "I shall have / I will have",
"i'm": "I am",
"i've": "I have",
"isn't": "is not",
"it'd": "it had / it would",
"it'd've": "it would have",
"it'll": "it shall / it will",
"it'll've": "it shall have / it will have",
"it's": "it has / it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she had / she would",
"she'd've": "she would have",
"she'll": "she shall / she will",
"she'll've": "she shall have / she will have",
"she's": "she has / she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so as / so is",
"that'd": "that would / that had",
"that'd've": "that would have",
"that's": "that has / that is",
"there'd": "there had / there would",
"there'd've": "there would have",
"there's": "there has / there is",
"they'd": "they had / they would",
"they'd've": "they would have",
"they'll": "they shall / they will",
"they'll've": "they shall have / they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we had / we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what shall / what will",
"what'll've": "what shall have / what will have",
"what're": "what are",
"what's": "what has / what is",
"what've": "what have",
"when's": "when has / when is",
"when've": "when have",
"where'd": "where did",
"where's": "where has / where is",
"where've": "where have",
"who'll": "who shall / who will",
"who'll've": "who shall have / who will have",
"who's": "who has / who is",
"who've": "who have",
"why's": "why has / why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you had / you would",
"you'd've": "you would have",
"you'll": "you shall / you will",
"you'll've": "you shall have / you will have",
"you're": "you are",
"you've": "you have"
}

In [4]:
nltk.download('stopwords') 
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jiaqidong/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jiaqidong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# Function to expand contractions using regex for word boundaries
def expand_apostrophe(text, apostrophe_dict):
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, apostrophe_dict.keys())) + r')\b')
    return pattern.sub(lambda match: apostrophe_dict[match.group(0)], text)

In [6]:
from nltk.tokenize import word_tokenize

# Remove stopwords and punctuation from input text
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = word_tokenize(text)
    filtered = [t for t in tokens if t not in stop_words and t.strip()]
    return ' '.join(filtered)

In [7]:
def clean_text(text):
    # Replace non-breaking spaces (\xa0) with a normal space
    text = text.replace("\xa0", " ")
    
    # Remove specific punctuation characters: +, /, @, -
    text = re.sub(r"[+/@]", "", text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [8]:
job_details = (
    df['job_ad_details']
      .apply(extract_text)
      .str.lower()
      .str.replace(r'\bjob title\b', '', regex=True) 
      .apply(lambda x: expand_apostrophe(x, apostrophe_dict))
      .apply(remove_stopwords)
      .apply(clean_text)
)
job_test_details = (
    df_test['job_ad_details']
      .apply(extract_text)
      .str.lower()
      .str.replace(r'\bjob title\b', '', regex=True) 
      .apply(lambda x: expand_apostrophe(x, apostrophe_dict))
      .apply(remove_stopwords)
      .apply(clean_text)
)

In [9]:
label_mapping = {
    "entry level": "entry", "junior": "entry", "graduate": "entry",
    "trainee": "entry", "student": "entry", "entry-level": "entry",
    "entry level assistant": "entry", "1st year apprentice": "entry",
    "2nd year apprentice": "entry", "apprentice": "entry",

    "intermediate": "intermediate", "mid-level": "intermediate",
    "associate": "intermediate", "assistant": "intermediate",
    "standard": "intermediate", "coordinator": "intermediate",
    "experienced": "intermediate", "qualified": "intermediate",
    "junior-intermediate": "intermediate", "experienced assistant": "intermediate",

    "senior": "senior", "lead": "senior", "senior associate": "senior",
    "senior lead": "senior", "senior/lead": "senior", "senior-executive": "senior",
    "senior assistant": "senior", "senior head": "senior",
    "mid-senior": "senior", "intermediate to senior": "senior",

    "head": "management", "director": "management", "assistant manager": "management",
    "assistant head": "management", "assistant director": "management",
    "associate director": "management", "regional head": "management",
    "middle-management": "management", "manager": "management",

    "executive": "executive", "chief": "executive", "principal": "executive",
    "general-manager": "executive", "owner": "executive", "owner-operator": "executive",
    "board": "executive", "supervisor": "executive", "second-in-command": "executive",
    "independent": "executive", "advanced": "executive",
}

In [10]:
# Map original labels to new labels using label_mapping dictionary
mapped_label = df['y_true'].apply(lambda x: label_mapping.get(x, 'other'))
mapped_label_test = df_test['y_true'].apply(lambda x: label_mapping.get(x, 'other'))

# Get unique labels and their counts from training data
unique_labels = mapped_label.unique().tolist()
label_counts = mapped_label.value_counts().to_dict()
print("Unique labels:", unique_labels)
print("Number of unique labels:", len(unique_labels))
print(label_counts)

Unique labels: ['intermediate', 'senior', 'management', 'entry', 'executive', 'other']
Number of unique labels: 6
{'intermediate': 1599, 'senior': 583, 'entry': 380, 'management': 103, 'executive': 55, 'other': 32}


In [24]:
from sklearn.metrics import f1_score, classification_report

predictions = []
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)
# Iterate over the job advertisement texts
for job_ad in job_details:
    result = classifier(job_ad, unique_labels)
    # The classifier returns a dictionary with keys "labels" and "scores".
    # 'labels' is sorted from highest to lowest score, so we take the first one.
    pred_label = result["labels"][0]
    predictions.append(pred_label)

# Optionally, you can store the predictions in the dataframe for further analysis.
df['predicted_seniority'] = predictions

# Step 4: Compute the F1 Score
# Using scikit-learn's f1_score function. Choose an averaging method (e.g., 'weighted', 'macro').
# Make sure that df_subset['y_true'] contains the correct ground-truth labels.
f1 = f1_score(mapped_label, predictions, average='weighted')
print("Weighted F1 Score:", f1)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Weighted F1 Score: 0.2616426696302603


In [25]:
from sklearn.metrics import classification_report
print(classification_report(mapped_label, predictions))

              precision    recall  f1-score   support

       entry       0.34      0.26      0.29       380
   executive       0.15      0.27      0.19        55
intermediate       0.71      0.12      0.20      1599
  management       0.08      0.57      0.13       103
       other       0.01      0.22      0.02        32
      senior       0.47      0.43      0.45       583

    accuracy                           0.22      2752
   macro avg       0.29      0.31      0.21      2752
weighted avg       0.56      0.22      0.26      2752



Spacy Model
    Process the job ad text and attempt to extract a seniority level based on
    the unique labels provided. The function tokenizes and lemmatizes the text,
    then for each label in unique_labels (which are assumed to be phrases),
    it checks if all the words in the label are present in the text.
    
    If one or more labels are found, it returns the first match (you can modify 
    the logic to choose a preferred match if needed). If no label is found,
    it returns a default value (e.g., "intermediate").

In [11]:
import spacy
nlp = spacy.load('en_core_web_sm')
def extract_seniority(text, nlp, unique_labels):
    # Process the text with spaCy.
    doc = nlp(text)
    # Get a list of lemmatized tokens for matching.
    lemmas = [token.lemma_ for token in doc]
    
    matched_labels = []
    for label in unique_labels:
        label_keywords = label.lower().split()
        if all(kw in lemmas for kw in label_keywords):
            matched_labels.append(label)
    
    if matched_labels:
        # Here, we simply return the first match.
        # You could add additional rules to prioritize certain labels.
        return matched_labels[0]
    else:
        # Return a default if no label is found.
        return "intermediate"
predicted_seniority = job_details.apply(lambda x: extract_seniority(x, nlp, unique_labels))

In [14]:
from sklearn.metrics import classification_report
print(classification_report(mapped_label, predicted_seniority))

              precision    recall  f1-score   support

       entry       0.48      0.08      0.13       380
   executive       0.08      0.05      0.07        55
intermediate       0.67      0.52      0.59      1599
  management       0.05      0.49      0.09       103
       other       0.00      0.00      0.00        32
      senior       0.58      0.40      0.47       583

    accuracy                           0.42      2752
   macro avg       0.31      0.26      0.22      2752
weighted avg       0.58      0.42      0.46      2752



In [27]:
job_ad = job_details.tolist()
job_ad_test = job_test_details.tolist()
labels = mapped_label.tolist()
labels_test = mapped_label_test.tolist()

le = LabelEncoder()
label_ids = le.fit_transform(labels)
label_ids_test = le.fit_transform(labels_test)
num_labels = len(le.classes_)

In [28]:
# Split data into training/validation sets with stratified sampling
train_texts, val_texts, train_labels, val_labels = train_test_split(
    job_ad, label_ids,
    test_size=0.3,
    random_state=42,
    stratify=label_ids
)

train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels})
val_dataset   = Dataset.from_dict({'text': val_texts,   'labels': val_labels})
test_dataset = Dataset.from_dict({'text': job_ad_test,   'labels': label_ids_test})
datasets = DatasetDict({'train': train_dataset, 'validation': val_dataset})

In [29]:
tokenizer = BartTokenizerFast.from_pretrained('facebook/bart-large-mnli')

# Define batch tokenization function for text processing
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

tokenized = datasets.map(tokenize_fn, batched=True)
tokenized.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
print(tokenized)

# Process test set with same tokenization
tokenized_test = test_dataset.map(tokenize_fn, batched=True)
tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/1926 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1926
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 826
    })
})


Map:   0%|          | 0/689 [00:00<?, ? examples/s]

In [30]:
# Initialize BART model for sequence classification task
model = BartForSequenceClassification.from_pretrained(
    'facebook/bart-large-mnli',
    num_labels=6,
    ignore_mismatched_sizes=True
)
data_collator = DataCollatorWithPadding(tokenizer)

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-large-mnli and are newly initialized because the shapes did not match:
- classification_head.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([6]) in the model instantiated
- classification_head.out_proj.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# Compute weighted F1 score for model evaluation
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]  # unwrap logits
    preds = np.argmax(logits, axis=1)
    labels = eval_pred.label_ids
    return {"weighted_f1": f1_score(labels, preds, average="weighted")}

In [32]:
import torch
from torch.utils.data import DataLoader
from transformers.optimization import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

# 2) DataLoaders
train_loader = DataLoader(tokenized['train'],
                          batch_size=8,
                          shuffle=True,
                          collate_fn=data_collator)
val_loader = DataLoader(tokenized['validation'],
                        batch_size=8,
                        collate_fn=data_collator)

# 3) Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

best_model_path = "./best_model"
best_val_loss = float("inf")

for epoch in range(1, num_epochs + 1):
    # Training loop (same as before)
    model.train()
    total_train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation loop
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"New best model saved at epoch {epoch}")
    model.save_pretrained("./final_model")
    tokenizer.save_pretrained("./final_model")

Train Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:50<00:00,  2.18it/s]


Epoch 1 | Train Loss: 1.1215 | Val Loss: 0.9089
✅ New best model saved at epoch 1


Train Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.21it/s]


Epoch 2 | Train Loss: 0.8001 | Val Loss: 0.8256
✅ New best model saved at epoch 2


Train Epoch 3: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.19it/s]


...

Epoch 8 | Train Loss: 0.0248 | Val Loss: 1.4114


Train Epoch 9: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:51<00:00,  2.17it/s]


Epoch 9 | Train Loss: 0.0128 | Val Loss: 1.4657


Train Epoch 10: 100%|████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.20it/s]


Epoch 10 | Train Loss: 0.0101 | Val Loss: 1.4845


In [33]:
from sklearn.metrics import accuracy_score

def evaluate_accuracy(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"Accuracy: {acc:.4f}")
    return acc

In [34]:
test_loader = DataLoader(
    tokenized_test,
    batch_size=16,           
    collate_fn=data_collator
)

In [35]:
# Reload the best model
model = BartForSequenceClassification.from_pretrained(best_model_path).to(device)

# Evaluate on validation set (or replace with test_loader if you have test data)
evaluate_accuracy(model, test_loader)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 87/87 [00:12<00:00,  6.88it/s]

✅ Accuracy: 0.6778


0.6777939042089985

In [36]:
# Reload the best model
model = BartForSequenceClassification.from_pretrained("./final_model").to(device)

# Evaluate on validation set (or replace with test_loader if you have test data)
evaluate_accuracy(model, test_loader)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 87/87 [00:12<00:00,  6.87it/s]

✅ Accuracy: 0.6967


0.6966618287373004

In [37]:
from sklearn.metrics import classification_report

def get_predictions_and_labels(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    return all_preds, all_labels

In [38]:
best_model = BartForSequenceClassification.from_pretrained(best_model_path).to(device)
best_preds, best_labels = get_predictions_and_labels(best_model, test_loader)

print("Classification Report (Best Model):")
print(classification_report(best_labels, best_preds, target_names=le.classes_))

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.


📊 Classification Report (Best Model):
              precision    recall  f1-score   support

       entry       0.62      0.55      0.58       111
   executive       0.00      0.00      0.00        17
intermediate       0.75      0.81      0.78       386
  management       0.25      0.04      0.06        27
       other       0.00      0.00      0.00        15
      senior       0.54      0.68      0.60       133

    accuracy                           0.68       689
   macro avg       0.36      0.35      0.34       689
weighted avg       0.64      0.68      0.65       689



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from collections import Counter
from deep_translator import GoogleTranslator

INPUT_CSV = "D:/comp-6713-industry-project/raw_data/seniority_labelled_development_set.csv"

In [2]:
# Version 1 for BERT: Only basic preprocessing, no label merging, no augmentation
OUTPUT_CSV = "./seniority_v1_baseline.csv"

def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

def expand_apostrophe(text):
    apostrophe_dict = {"don't": "do not", "you're": "you are", "i'm": "i am", "it's": "it is"}
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, apostrophe_dict.keys())) + r')\b')
    return pattern.sub(lambda match: apostrophe_dict[match.group(0)], text)

print("Processing V1 baseline data")

df = pd.read_csv(INPUT_CSV)
df['text'] = df['job_ad_details'].apply(extract_text).str.lower().apply(expand_apostrophe)
df['label'] = df['y_true']

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df[['text', 'label']].to_csv(OUTPUT_CSV, index=False)
print(f"V1 baseline data saved to {OUTPUT_CSV}")

Processing V1 baseline data
V1 baseline data saved to ./seniority_v1_baseline.csv


In [3]:
# Version 2 for BERT: Add label merging and rare class filtering
OUTPUT_CSV = "./seniority_v2_label_merge.csv"

label_merge_dict = {
    "entry level": "entry", "entry-level": "entry", "graduate": "entry", "student": "entry",
    "junior": "intermediate", "intermediate": "intermediate",
    "assistant": "assistant", "assistant director": "assistant",
    "mid-level": "mid", "associate": "mid",
    "senior": "senior", "lead": "senior",
    "manager": "manager",
    "director": "director", "executive": "director",
    "specialist": "specialist", "supervisor": "specialist",
    "experienced": "experienced"
}

def merge_label(label):
    return label_merge_dict.get(label.strip().lower(), "other")

def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

def expand_apostrophe(text):
    apostrophe_dict = {"don't": "do not", "you're": "you are", "i'm": "i am", "it's": "it is"}
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, apostrophe_dict.keys())) + r')\b')
    return pattern.sub(lambda match: apostrophe_dict[match.group(0)], text)

print("Processing V2 label merge data")
df = pd.read_csv(INPUT_CSV)
df['text'] = df['job_ad_details'].apply(extract_text).str.lower().apply(expand_apostrophe)
df['label'] = df['y_true'].apply(merge_label)

# Filter very rare classes (appear < 10 times)
label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 10].index.tolist()
df = df[df['label'].isin(valid_labels)].reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df[['text', 'label']].to_csv(OUTPUT_CSV, index=False)
print(f"V2 label merge data saved to {OUTPUT_CSV}")

Processing V2 label merge data
V2 label merge data saved to ./seniority_v2_label_merge.csv


In [4]:
# eda.py (official full version)
# Source: https://github.com/jasonwei20/eda_nlp

import random
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# Synonym replacement
def synonym_replacement(words, n):
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word not in stop_words]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(list(synonyms))
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return new_words

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for l in syn.lemmas():
            synonym = l.name().replace("_", " ").replace("-", " ").lower()
            synonym = "".join([char for char in synonym if char in ' qwertyuiopasdfghjklzxcvbnm'])
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)

# Random insertion
def random_insertion(words, n):
    new_words = words.copy()
    for _ in range(n):
        add_word(new_words)
    return new_words

def add_word(new_words):
    synonyms = []
    counter = 0
    while len(synonyms) < 1 and counter < 10:
        random_word = new_words[random.randint(0, len(new_words)-1)]
        synonyms = get_synonyms(random_word)
        counter += 1
    if len(synonyms) > 0:
        random_synonym = synonyms[0]
        random_idx = random.randint(0, len(new_words)-1)
        new_words.insert(random_idx, random_synonym)

# Random swap
def random_swap(words, n):
    new_words = words.copy()
    for _ in range(n):
        new_words = swap_word(new_words)
    return new_words

def swap_word(new_words):
    if len(new_words) < 2:
        return new_words
    idx1, idx2 = random.sample(range(len(new_words)), 2)
    new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return new_words

# Random deletion
def random_deletion(words, p):
    if len(words) == 1:
        return words
    return [word for word in words if random.uniform(0, 1) > p] or [random.choice(words)]

# Main EDA function
def eda(sentence, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, p_rd=0.1, num_aug=4):
    sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)
    words = sentence.split()
    num_words = len(words)
    augmented_sentences = []
    n_sr = max(1, int(alpha_sr * num_words))
    n_ri = max(1, int(alpha_ri * num_words))
    n_rs = max(1, int(alpha_rs * num_words))

    for _ in range(num_aug):
        a_words = words.copy()
        if alpha_sr > 0:
            a_words = synonym_replacement(a_words, n_sr)
        if alpha_ri > 0:
            a_words = random_insertion(a_words, n_ri)
        if alpha_rs > 0:
            a_words = random_swap(a_words, n_rs)
        if p_rd > 0:
            a_words = random_deletion(a_words, p_rd)
        augmented_sentences.append(' '.join(a_words))

    return augmented_sentences

[nltk_data] Downloading package wordnet to C:\Users\Song
[nltk_data]     Yidong\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Song
[nltk_data]     Yidong\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Song
[nltk_data]     Yidong\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# Version 3 for BERT: Label merging + EDA + Back-Translation augmentation (final version)
OUTPUT_CSV = "./seniority_v3_fullaug.csv"
TARGET_SAMPLES_PER_CLASS = 60
EDA_PER_SAMPLE = 5
BT_PER_CLASS = 3

label_merge_dict = {
    "entry level": "entry", "entry-level": "entry", "graduate": "entry", "student": "entry",
    "junior": "intermediate", "intermediate": "intermediate",
    "assistant": "assistant", "assistant director": "assistant",
    "mid-level": "mid", "associate": "mid",
    "senior": "senior", "lead": "senior",
    "manager": "manager",
    "director": "director", "executive": "director",
    "specialist": "specialist", "supervisor": "specialist",
    "experienced": "experienced"
}

def merge_label(label):
    return label_merge_dict.get(label.strip().lower(), "other")

def extract_text(html):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

def expand_apostrophe(text):
    import re
    apostrophe_dict = {"don't": "do not", "you're": "you are", "i'm": "i am", "it's": "it is"}
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, apostrophe_dict.keys())) + r')\b')
    return pattern.sub(lambda match: apostrophe_dict[match.group(0)], text)

print("Processing V3 full augmentation data")
df = pd.read_csv(INPUT_CSV)
df['text'] = df['job_ad_details'].apply(extract_text).str.lower().apply(expand_apostrophe)
df['label'] = df['y_true'].apply(merge_label)

df_clean = df[['text', 'label']]
label_counts = df_clean['label'].value_counts()
underrepresented_labels = label_counts[label_counts < TARGET_SAMPLES_PER_CLASS].index.tolist()

# Augmentation
aug_data = []
for label in underrepresented_labels:
    samples = df_clean[df_clean['label'] == label]
    needed = TARGET_SAMPLES_PER_CLASS - len(samples)

    # Back-translation
    bt_samples = samples.sample(min(BT_PER_CLASS, len(samples)), random_state=42)
    for _, row in bt_samples.iterrows():
        try:
            zh = GoogleTranslator(source='en', target='zh-CN').translate(row['text'])
            back = GoogleTranslator(source='zh-CN', target='en').translate(zh)
            aug_data.append({"text": back, "label": label})
        except:
            continue

    # EDA
    while len([x for x in aug_data if x['label'] == label]) < needed:
        for _, row in samples.iterrows():
            edas = eda(row['text'], num_aug=EDA_PER_SAMPLE, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, p_rd=0.1)
            for aug_text in edas:
                aug_data.append({"text": aug_text, "label": label})
                if len([x for x in aug_data if x['label'] == label]) >= needed:
                    break

final_df = pd.concat([df_clean, pd.DataFrame(aug_data)], ignore_index=True)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"V3 full augmentation data saved to {OUTPUT_CSV}")

Processing V3 full augmentation data
V3 full augmentation data saved to ./seniority_v3_fullaug.csv


In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from tqdm import tqdm
import os

# Config
MODEL_SAVE_PATH = "./best_model_dynamic"
LABEL_CLASS_PATH = "label_classes.npy"
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
VAL_RATIO = 0.2
SEED = 42

class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_and_eval(train_csv, if_output_cm):
    # Load and prepare data
    df = pd.read_csv(train_csv)
    df = df.groupby('label').filter(lambda x: len(x) >= 2)
    texts = df['text'].tolist()
    labels_raw = df['label'].tolist()
    
    # Label encoding
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(labels_raw)
    np.save(LABEL_CLASS_PATH, label_encoder.classes_)
    
    # Split dataset
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=VAL_RATIO, random_state=SEED, stratify=labels
    )
    
    print("\nClass distribution after split:")
    print(pd.Series(train_labels).value_counts().sort_index())
    
    # Class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    
    # Model setup
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(label_encoder.classes_)
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    class_weights_tensor = class_weights_tensor.to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    # Prepare dataloaders
    train_dataset = JobDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_dataset = JobDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    
    # Training variables
    best_val_loss = float('inf')
    best_preds = None
    best_true = None
    history = []
    
    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        
        # Training phase
        for batch in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = torch.nn.functional.cross_entropy(
                outputs.logits, 
                labels, 
                weight=class_weights_tensor
            )
            
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        epoch_preds = []
        epoch_true = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask)
                loss = torch.nn.functional.cross_entropy(
                    outputs.logits, 
                    labels, 
                    weight=class_weights_tensor
                )
                
                total_val_loss += loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                epoch_preds.extend(preds.cpu().numpy())
                epoch_true.extend(labels.cpu().numpy())
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = accuracy_score(epoch_true, epoch_preds)
        val_f1 = f1_score(epoch_true, epoch_preds, average='macro')
        
        # Track best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_preds = epoch_preds.copy()
            best_true = epoch_true.copy()
            model.save_pretrained(MODEL_SAVE_PATH)
            tokenizer.save_pretrained(MODEL_SAVE_PATH)
            print(f"New best model saved at epoch {epoch+1}")
        
        # Record history
        history.append({
            "epoch": epoch+1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1
        })
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")
    
    # Final evaluation
    print("\n=== Final Evaluation Results ===")
    print(f"Best validation loss: {best_val_loss:.4f}")
    
    # Classification report
    final_report = classification_report(
        best_true,
        best_preds,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        digits=2,
        zero_division=0
    )
    print("\nClassification Report:")
    print(final_report)
    
    # Confusion matrix
    if if_output_cm:
        cm = confusion_matrix(
            best_true,
            best_preds,
            labels=np.arange(len(label_encoder.classes_))
        )
        cm_df = pd.DataFrame(
            cm,
            index=label_encoder.classes_,
            columns=label_encoder.classes_
        )
        print("\nConfusion Matrix:")
        print(cm_df.to_string())
    
    # Save training history
    os.makedirs("./logs", exist_ok=True)
    pd.DataFrame(history).to_csv("./logs/training_history.csv", index=False)
    print("\nTraining history saved to ./logs/training_history.csv")

In [2]:
# Hard to output a large scale confusion matrix with pretty syntax
train_and_eval("./seniority_v1_baseline.csv", False)


Class distribution after split:
0       2
1       2
2      12
3     106
4      14
5       2
6       7
7       2
8       6
9       9
10    198
11      8
12      2
13     12
14    672
15      3
16     20
17      2
18     59
19    470
20      2
21     45
22     93
23     10
24      6
25      2
26      2
27     13
28      3
29    361
30      2
31      2
32      2
33      2
34      5
35      2
36      2
37      3
38     16
Name: count, dtype: int64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Song Yidong\AppData\Roaming\Python\Python312\site-packages\transformers\optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Train Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 137/137 [00:27<00:00,  5.01it/s]


New best model saved at epoch 1
Epoch 1 | Train Loss: 3.6004 | Val Loss: 3.5210 | Val Acc: 0.2143 | Val Macro F1: 0.0160


Train Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 137/137 [00:25<00:00,  5.28it/s]


New best model saved at epoch 2
Epoch 2 | Train Loss: 3.5010 | Val Loss: 3.4841 | Val Acc: 0.2161 | Val Macro F1: 0.0161


...

Epoch 18 | Train Loss: 0.3792 | Val Loss: 3.1647 | Val Acc: 0.5147 | Val Macro F1: 0.2066


Train Epoch 19: 100%|████████████████████████████████████████████████████████████████| 137/137 [00:26<00:00,  5.17it/s]


Epoch 19 | Train Loss: 0.3206 | Val Loss: 3.1922 | Val Acc: 0.5000 | Val Macro F1: 0.2091


Train Epoch 20: 100%|████████████████████████████████████████████████████████████████| 137/137 [00:26<00:00,  5.19it/s]


Epoch 20 | Train Loss: 0.2710 | Val Loss: 3.3782 | Val Acc: 0.5201 | Val Macro F1: 0.2111

=== Final Evaluation Results ===
Best validation loss: 2.7495

Classification Report:
                             precision    recall  f1-score   support

        4th year apprentice       0.00      0.00      0.00         0
                   advanced       0.00      0.00      0.00         0
                 apprentice       0.40      0.67      0.50         3
                  assistant       0.40      0.63      0.49        27
                  associate       0.29      0.50      0.36         4
                      board       0.00      0.00      0.00         1
                      chief       0.00      0.00      0.00         2
                coordinator       0.00      0.00      0.00         0
                     deputy       0.00      0.00      0.00         2
                   director       0.00      0.00      0.00         2
                entry level       0.24      0.72      0.37     

In [3]:
train_and_eval("./seniority_v2_label_merge.csv", True)


Class distribution after split:
0    107
1     21
2    229
3    672
4    515
5      9
6     20
7    165
8    455
9      8
Name: count, dtype: int64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Song Yidong\AppData\Roaming\Python\Python312\site-packages\transformers\optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Train Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 138/138 [00:26<00:00,  5.17it/s]


New best model saved at epoch 1
Epoch 1 | Train Loss: 2.2766 | Val Loss: 2.1926 | Val Acc: 0.2359 | Val Macro F1: 0.0739


Train Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 138/138 [00:26<00:00,  5.25it/s]


Epoch 2 | Train Loss: 2.1704 | Val Loss: 2.2416 | Val Acc: 0.2668 | Val Macro F1: 0.0761


...

Epoch 18 | Train Loss: 0.0140 | Val Loss: 3.4274 | Val Acc: 0.5118 | Val Macro F1: 0.3368


Train Epoch 19: 100%|████████████████████████████████████████████████████████████████| 138/138 [00:26<00:00,  5.22it/s]


Epoch 19 | Train Loss: 0.0109 | Val Loss: 3.3871 | Val Acc: 0.5191 | Val Macro F1: 0.3439


Train Epoch 20: 100%|████████████████████████████████████████████████████████████████| 138/138 [00:26<00:00,  5.22it/s]


Epoch 20 | Train Loss: 0.0147 | Val Loss: 3.2932 | Val Acc: 0.5209 | Val Macro F1: 0.3722

=== Final Evaluation Results ===
Best validation loss: 1.8135

Classification Report:
              precision    recall  f1-score   support

   assistant       0.47      0.56      0.51        27
    director       0.15      0.60      0.24         5
       entry       0.56      0.39      0.46        57
 experienced       0.49      0.49      0.49       168
intermediate       0.43      0.29      0.35       129
     manager       0.00      0.00      0.00         3
         mid       0.07      0.40      0.12         5
       other       0.21      0.10      0.13        41
      senior       0.41      0.56      0.47       114
  specialist       0.00      0.00      0.00         2

    accuracy                           0.42       551
   macro avg       0.28      0.34      0.28       551
weighted avg       0.43      0.42      0.42       551


Confusion Matrix:
              assistant  director  entry  exp

In [4]:
train_and_eval("./seniority_v3_fullaug.csv", True)


Class distribution after split:
0    107
1     63
2    229
3    672
4    515
5     51
6     62
7    165
8    455
9     48
Name: count, dtype: int64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Song Yidong\AppData\Roaming\Python\Python312\site-packages\transformers\optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Train Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 148/148 [00:28<00:00,  5.20it/s]


New best model saved at epoch 1
Epoch 1 | Train Loss: 2.1322 | Val Loss: 1.9824 | Val Acc: 0.1858 | Val Macro F1: 0.1645


Train Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 148/148 [00:28<00:00,  5.15it/s]


New best model saved at epoch 2
Epoch 2 | Train Loss: 1.8382 | Val Loss: 1.7088 | Val Acc: 0.2956 | Val Macro F1: 0.3719


...

Epoch 18 | Train Loss: 0.0069 | Val Loss: 2.3026 | Val Acc: 0.5777 | Val Macro F1: 0.6231


Train Epoch 19: 100%|████████████████████████████████████████████████████████████████| 148/148 [00:28<00:00,  5.12it/s]


Epoch 19 | Train Loss: 0.0052 | Val Loss: 2.3064 | Val Acc: 0.5777 | Val Macro F1: 0.6353


Train Epoch 20: 100%|████████████████████████████████████████████████████████████████| 148/148 [00:28<00:00,  5.12it/s]


Epoch 20 | Train Loss: 0.0047 | Val Loss: 2.3363 | Val Acc: 0.5878 | Val Macro F1: 0.6306

=== Final Evaluation Results ===
Best validation loss: 1.1987

Classification Report:
              precision    recall  f1-score   support

   assistant       0.56      0.52      0.54        27
    director       0.62      0.81      0.70        16
       entry       0.51      0.47      0.49        57
 experienced       0.74      0.54      0.62       168
intermediate       0.47      0.53      0.50       129
     manager       0.69      0.75      0.72        12
         mid       0.79      0.69      0.73        16
       other       0.27      0.22      0.24        41
      senior       0.50      0.70      0.59       114
  specialist       1.00      0.58      0.74        12

    accuracy                           0.55       592
   macro avg       0.61      0.58      0.59       592
weighted avg       0.57      0.55      0.55       592


Confusion Matrix:
              assistant  director  entry  exp